In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🚀 Section 4: The Hidden Cost — What the KV Cache Does to Your GPU

*Part of the Vizuara AI/ML Series*
*Estimated time: 25–35 minutes*

## 1. Why Does This Matter?

Every time you chat with an LLM — whether you are asking it to summarise a long document, continue a story, or write code — the model generates one token at a time. That token generation step is where your GPU spends most of its life. And the KV cache is the trick that makes it fast enough to feel interactive.

But here is the tension no one talks about in tutorials: **the KV cache, which eliminates enormous amounts of arithmetic, simultaneously creates a different kind of pressure on your hardware.** It moves the bottleneck. Understanding exactly where it moves, and why, is the difference between someone who uses LLM systems and someone who can actually improve them.

By the end of this notebook, you will have:

- A working, measurable understanding of the **roofline model** — the hardware's two fundamental limits
- A clear picture of **arithmetic intensity** and why it is the right lens for thinking about LLM inference
- Concrete numbers comparing the naive inference loop against KV-cached inference
- An interactive roofline plot that places real LLM workloads on the hardware map

Here is a preview of what we will produce — a roofline diagram annotated with where naive inference, KV-cached inference, and matrix-heavy training all live:

In [ ]:
# ── Preview: the final annotated roofline plot we will build together ──
# (Run all cells in order — this preview cell will make full sense by Section 8)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_facecolor("#0f0f1a")
fig.patch.set_facecolor("#0f0f1a")

intensity = np.logspace(-2, 4, 500)
peak_compute = 312          # A100 Tensor Core TFLOP/s (BF16)
peak_bandwidth = 2.0        # A100 HBM bandwidth  TB/s  → TFLOP/byte = 2 FLOP/byte * 1e12/1e12

roofline = np.minimum(peak_bandwidth * intensity, peak_compute)

ax.loglog(intensity, roofline, color="#00d4ff", lw=3, label="Roofline")
ax.axhline(peak_compute, color="#00d4ff", lw=1, ls="--", alpha=0.4)

for point, label, color in [
    (0.05,  "Naive inference\n(re-reads everything)", "#ff6b6b"),
    (0.3,   "KV-cached inference\n(decode step)",     "#ffa94d"),
    (100,   "Training / prefill\n(compute-bound)",    "#69db7c"),
]:
    y = min(peak_bandwidth * point, peak_compute)
    ax.scatter([point], [y], s=180, color=color, zorder=5)
    ax.annotate(label, (point, y), textcoords="offset points",
                xytext=(10, 8), fontsize=9, color=color,
                fontweight="bold")

ax.set_xlabel("Arithmetic Intensity  (FLOP / byte)", color="white", fontsize=12)
ax.set_ylabel("Achieved Performance  (TFLOP/s)", color="white", fontsize=12)
ax.set_title("Roofline Model — Where Does LLM Inference Live?",
             color="white", fontsize=14, fontweight="bold")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_edgecolor("#333355")
ax.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=10)
plt.tight_layout()
plt.savefig("roofline_preview.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Roofline preview rendered — we will build every number behind this plot together.")

## 2. Building Intuition

Before we write a single equation, let us make sure the underlying ideas are crystal clear.

### The GPU Has Two Masters

A GPU is, at its heart, an arithmetic machine. It performs multiply-and-add operations at staggering speed — hundreds of trillions of floating-point operations per second on modern hardware. But here is the thing: **the arithmetic units cannot work on data they do not have.** Data lives in VRAM (the GPU's main memory), and it must travel across a physical bus into the chip before the chip can touch it. That journey has a speed limit called **memory bandwidth**, measured in bytes per second.

So every computation lives under two possible ceilings:

1. **Compute ceiling** — how many arithmetic operations the chip can do per second (FLOP/s)
2. **Bandwidth ceiling** — how many bytes per second can travel from VRAM to the chip

Whichever ceiling you hit first is your bottleneck.

### The Shipyard Analogy

Imagine the GPU chip is a shipyard that can assemble 1,000 ships per second — blazing fast. Raw materials (data) arrive by delivery trucks from a warehouse (VRAM). Each truck carries 100 units of material, and only one truck can arrive per second.

No matter how fast the shipyard is, it can only build 100 ships per second, because that is all the material it receives. The shipyard sits **idle for 90% of the day, waiting for trucks.** This is the **memory-bound regime**.

Now imagine trucks arrive 20 at a time — 2,000 units per second. The shipyard is now the bottleneck; it can only absorb 1,000 per second. This is the **compute-bound regime**.

### Where Does LLM Inference Fit?

Consider the KV-cached decode step — generating one new token. The model has to:
- Load the model weights from VRAM (large: hundreds of millions of parameters)
- Load the KV cache for the current sequence (large: grows with context length)
- Perform a relatively small amount of arithmetic (one token's worth)

The ratio of arithmetic to data movement is **very low**. We are loading a huge amount of data to do a small amount of computation. We are deep in the memory-bound region — our shipyard (GPU compute) sits hungry, waiting for trucks (memory bandwidth).

The KV cache makes inference *fast relative to naive re-computation*, but it does so by trading compute pressure for **memory pressure**. That is the hidden cost.

### 🤔 Think About This

Before moving on, sit with this question:

> The KV cache removes the need to recompute keys and values for past tokens — that sounds strictly better. So why would it make anything worse? What would have to be true about the hardware for eliminating arithmetic to hurt performance?

*Hint: think about what the chip was doing during all that arithmetic in the naive case — was the memory bus as busy as the chip?*

## 3. The Mathematics

Let us now make the intuitions precise. We will derive the key quantities step by step.

### 3.1 Arithmetic Intensity

The central quantity of the roofline model is **arithmetic intensity**, defined as:

$$I = \frac{\text{Number of floating-point operations (FLOP)}}{\text{Bytes of data moved from VRAM}}$$

This equation says: for every byte we pull from memory, how many arithmetic operations do we perform? If $I$ is large, we are doing a lot of work per byte — we are compute-bound. If $I$ is small, we are doing very little work per byte — we are memory-bound.

Computationally, this means: count all the multiplications and additions your algorithm performs, divide by all the bytes you read and write to/from VRAM. The units are FLOP/byte.

### 3.2 The Roofline Bound

Given hardware with peak compute $\pi$ (FLOP/s) and peak bandwidth $\beta$ (bytes/s), achieved performance $P$ is bounded by:

$$P \leq \min\!\left(\pi,\; \beta \cdot I\right)$$

This equation says: your performance is capped at either the hardware's peak arithmetic throughput *or* the bandwidth times your arithmetic intensity — whichever is smaller.

Computationally, this means: if $I < \pi / \beta$, bandwidth is your limit and performance scales linearly with $I$. If $I \geq \pi / \beta$, compute is your limit and performance plateaus.

The **ridge point** $I^* = \pi / \beta$ is the arithmetic intensity at which you transition from memory-bound to compute-bound.

$$I^* = \frac{\pi}{\beta}$$

For an NVIDIA A100 with BF16 Tensor Cores: $\pi = 312$ TFLOP/s, $\beta = 2.0$ TB/s, so $I^* \approx 156$ FLOP/byte. To be compute-bound you need to perform 156 arithmetic operations for every single byte you move. That is a high bar.

### 3.3 Arithmetic Intensity of a Linear Layer

A linear layer $Y = XW$ with input $X \in \mathbb{R}^{B \times d}$ and weight $W \in \mathbb{R}^{d \times d}$ performs:

$$\text{FLOP} = 2 \cdot B \cdot d^2$$

The factor of 2 accounts for multiply-and-add pairs. Bytes moved (loading weights + input, writing output) in FP16 (2 bytes each):

$$\text{Bytes} = 2\left(B \cdot d + d^2 + B \cdot d\right) \approx 2\,d^2 \quad (\text{when } B \ll d)$$

For large batch size $B$, the weight matrix $d^2$ is amortised over many inputs and intensity is high. For $B = 1$ (single-token decode):

$$I_{\text{decode}} = \frac{2 \cdot 1 \cdot d^2}{2(d^2 + 2d)} \approx \frac{2d^2}{2d^2} = 1 \;\text{FLOP/byte}$$

One FLOP per byte. The A100 ridge point is 156 FLOP/byte. We are operating at **1/156th of the point needed to be compute-bound.** The chip is almost entirely idle — waiting for data.

## 4. Let's Build It — Component by Component

We will build the roofline model interactively, measuring real arithmetic intensities as we go.

### 4.1 Roofline Model Foundation

Let us first implement the core roofline calculation and visualisation as a reusable class.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import time
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

# ── Hardware specification dataclass ──────────────────────────────────────────
@dataclass
class HardwareSpec:
    """Represents the roofline parameters of a GPU."""
    name: str
    peak_compute_tflops: float   # Tensor Core FLOP/s in BF16/FP16 (TFLOP/s)
    peak_bandwidth_tbs: float    # HBM bandwidth (TB/s)

    @property
    def ridge_point(self) -> float:
        """The arithmetic intensity (FLOP/byte) at which we transition regimes."""
        # Convert units: TFLOP/s  ÷  TB/s  =  FLOP/byte  (the 1e12 cancels)
        return self.peak_compute_tflops / self.peak_bandwidth_tbs

    def achieved_performance(self, intensity: np.ndarray) -> np.ndarray:
        """
        Apply the roofline bound:  P = min(peak_compute, bandwidth * intensity)
        intensity is in FLOP/byte, output is TFLOP/s.
        """
        bandwidth_bound = self.peak_bandwidth_tbs * intensity   # TFLOP/s
        return np.minimum(self.peak_compute_tflops, bandwidth_bound)

# ── Canonical GPU specs ────────────────────────────────────────────────────────
A100 = HardwareSpec("NVIDIA A100 (BF16)", peak_compute_tflops=312, peak_bandwidth_tbs=2.0)
T4   = HardwareSpec("NVIDIA T4 (FP16)",   peak_compute_tflops=65,  peak_bandwidth_tbs=0.3)

print("="*55)
print(f"  {A100.name}")
print(f"  Peak Compute   : {A100.peak_compute_tflops:>6.0f} TFLOP/s")
print(f"  Peak Bandwidth : {A100.peak_bandwidth_tbs:>6.1f} TB/s")
print(f"  Ridge Point    : {A100.ridge_point:>6.1f} FLOP/byte")
print()
print(f"  {T4.name}")
print(f"  Peak Compute   : {T4.peak_compute_tflops:>6.0f} TFLOP/s")
print(f"  Peak Bandwidth : {T4.peak_bandwidth_tbs:>6.1f} TB/s")
print(f"  Ridge Point    : {T4.ridge_point:>6.1f} FLOP/byte")
print("="*55)
print(f"\n  To be compute-bound on A100, you need ≥ {A100.ridge_point:.0f} FLOP/byte.")
print(f"  A single-token decode gives us ~1 FLOP/byte.")
print(f"  That is {A100.ridge_point/1:.0f}× below the ridge point. 🚨")

In [ ]:
# ── 📊 Visualisation checkpoint: the bare roofline ──────────────────────────
def plot_roofline(hw: HardwareSpec, ax=None, color="#00d4ff", label=None):
    """Draw the roofline curve for a given hardware spec."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))

    intensity = np.logspace(-2, 4, 1000)
    perf      = hw.achieved_performance(intensity)

    ax.loglog(intensity, perf, color=color, lw=2.5,
              label=label or hw.name)

    # Annotate ridge point
    ax.axvline(hw.ridge_point, color=color, lw=1, ls=":", alpha=0.6)
    ax.text(hw.ridge_point * 1.15, hw.peak_compute_tflops * 0.6,
            f"Ridge: {hw.ridge_point:.0f}", color=color, fontsize=8,
            rotation=90, va="top")

    # Shade regions
    mem_mask  = intensity <= hw.ridge_point
    comp_mask = intensity >= hw.ridge_point
    ax.fill_between(intensity, perf, alpha=0.07, color=color)
    return ax

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_facecolor("#0f0f1a"); fig.patch.set_facecolor("#0f0f1a")

plot_roofline(A100, ax=ax, color="#00d4ff", label="A100 BF16")
plot_roofline(T4,   ax=ax, color="#a78bfa", label="T4 FP16")

# Axis formatting
for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
             ax.get_xticklabels() + ax.get_yticklabels()):
    item.set_color("white")
ax.tick_params(colors="white")
for spine in ax.spines.values(): spine.set_edgecolor("#333355")

ax.set_xlabel("Arithmetic Intensity  (FLOP / byte)", fontsize=12)
ax.set_ylabel("Achieved Performance  (TFLOP/s)",      fontsize=12)
ax.set_title("The Roofline Model — Two GPU Profiles",  fontsize=14, fontweight="bold", color="white")
ax.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=10)

# Label regions on A100
ax.text(0.025, 180, "Memory-Bound\nregion", color="#00d4ff", fontsize=9, alpha=0.8)
ax.text(500,   180, "Compute-Bound\nregion", color="#00d4ff", fontsize=9, alpha=0.8)

plt.tight_layout()
plt.savefig("roofline_base.png", dpi=120, bbox_inches="tight")
plt.show()
print("📊 Base roofline rendered. Next: we compute where LLM workloads actually land.")

### 4.2 Computing Arithmetic Intensity for LLM Operations

Now let us build a calculator that takes model parameters and computes the arithmetic intensity for the key LLM operations. We will compare three scenarios side-by-side.

In [ ]:
@dataclass
class LLMWorkload:
    """
    Describes one operation during LLM inference and computes its arithmetic intensity.

    Parameters
    ----------
    name        : Human-readable label
    flop        : Total floating-point operations
    bytes_moved : Total bytes read from / written to VRAM
    color       : Plot colour
    """
    name:        str
    flop:        float
    bytes_moved: float
    color:       str = "#ffffff"

    @property
    def intensity(self) -> float:
        return self.flop / self.bytes_moved

    def summary(self):
        print(f"  {self.name}")
        print(f"    FLOP        : {self.flop/1e9:>10.2f} GFLOP")
        print(f"    Bytes moved : {self.bytes_moved/1e9:>10.2f} GB")
        print(f"    Intensity   : {self.intensity:>10.2f} FLOP/byte")
        regime = "compute-bound ✅" if self.intensity >= A100.ridge_point else "memory-bound ⚠️"
        print(f"    Regime (A100): {regime}")
        print()

# ── Helper: compute FLOP and bytes for key LLM operations ─────────────────────

def linear_layer_workload(name, batch, d_model, d_out, dtype_bytes=2, color="#fff"):
    """
    Y = X @ W     where X: [batch, d_model], W: [d_model, d_out]

    FLOP  = 2 * batch * d_model * d_out   (one fused multiply-add = 2 FLOP)
    Bytes = dtype_bytes * (batch*d_model + d_model*d_out + batch*d_out)
              read input   read weights         write output
    """
    flop  = 2 * batch * d_model * d_out
    bytes_in  = batch  * d_model          # input activations
    bytes_w   = d_model * d_out           # weight matrix
    bytes_out = batch  * d_out            # output
    total_bytes = dtype_bytes * (bytes_in + bytes_w + bytes_out)
    return LLMWorkload(name, flop, total_bytes, color)

def attention_workload(name, batch, seq_len, n_heads, d_head, dtype_bytes=2, color="#fff"):
    """
    QK^T softmax V attention.

    FLOP  ≈ 4 * batch * n_heads * seq_len^2 * d_head
              (QK matmul + softmax-scale + AV matmul, rough count)
    Bytes = Q, K, V tensors + attention scores + output
    """
    flop  = 4 * batch * n_heads * seq_len**2 * d_head
    bytes_qkv   = 3 * batch * n_heads * seq_len * d_head
    bytes_scores = batch * n_heads * seq_len**2
    bytes_out    = batch * n_heads * seq_len * d_head
    total_bytes  = dtype_bytes * (bytes_qkv + bytes_scores + bytes_out)
    return LLMWorkload(name, flop, total_bytes, color)

# ── Concrete model: GPT-2 scale (d=768, 12 heads, head_dim=64) ────────────────
D     = 768
N_H   = 12
D_H   = D // N_H   # 64

print("=" * 60)
print("  Workload Analysis — GPT-2 scale (d=768, 12 heads)")
print("=" * 60)

workloads = [
    # Prefill: process the full prompt of 512 tokens at once (batch=1)
    linear_layer_workload("Prefill: Linear layer  (B=1, S=512)",
                          batch=512, d_model=D, d_out=D, color="#69db7c"),
    attention_workload   ("Prefill: Attention      (B=1, S=512)",
                          batch=1, seq_len=512, n_heads=N_H, d_head=D_H, color="#74c0fc"),

    # Decode (KV-cached): process 1 new token, attend over 512-token KV cache
    linear_layer_workload("Decode:  Linear layer  (B=1, S=1)",
                          batch=1,   d_model=D, d_out=D, color="#ffa94d"),
    attention_workload   ("Decode:  Attention      (B=1, S=1, KV_len=512)",
                          batch=1, seq_len=1, n_heads=N_H, d_head=D_H, color="#ff6b6b"),
]

for w in workloads:
    w.summary()

In [ ]:
# ── 📊 Bar chart: intensity side-by-side ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
ax.set_facecolor("#0f0f1a"); fig.patch.set_facecolor("#0f0f1a")

labels     = [w.name for w in workloads]
intensities = [w.intensity for w in workloads]
colors      = [w.color for w in workloads]

bars = ax.barh(labels, intensities, color=colors, alpha=0.85, edgecolor="#333355")

# Ridge point reference line (A100)
ax.axvline(A100.ridge_point, color="#00d4ff", lw=2, ls="--",
           label=f"A100 ridge point ({A100.ridge_point:.0f} FLOP/byte)")

# Annotate bars with values
for bar, val in zip(bars, intensities):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}", va="center", ha="left", color="white", fontsize=9)

ax.set_xlabel("Arithmetic Intensity (FLOP / byte)", color="white", fontsize=12)
ax.set_title("Arithmetic Intensity: Prefill vs Decode Operations",
             color="white", fontsize=13, fontweight="bold")
ax.tick_params(colors="white")
for spine in ax.spines.values(): spine.set_edgecolor("#333355")
ax.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=10)

plt.tight_layout()
plt.savefig("intensity_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("📊 Notice: decode operations are far below the ridge point — solidly memory-bound.")

### 4.3 How Sequence Length and Batch Size Shift the Regime

The arithmetic intensity of the decode step is not fixed — it shifts with batch size (more tokens decoded in parallel) and sequence length (more KV cache to load). Let us map out this landscape.

In [ ]:
def decode_linear_intensity(batch: int, d: int = 768) -> float:
    """
    Arithmetic intensity for a single linear layer during KV-cached decode.
    batch: number of sequences decoded in parallel
    """
    flop        = 2 * batch * d * d
    bytes_moved = 2 * (batch * d + d * d + batch * d)   # FP16 = 2 bytes
    return flop / bytes_moved

def decode_attention_intensity(batch: int, kv_len: int,
                                n_heads: int = 12, d_head: int = 64) -> float:
    """
    Arithmetic intensity for the attention step during KV-cached decode.
    The KV cache must be read in full for every decode step.
    """
    # One query token attends over kv_len cached tokens
    flop         = 2 * batch * n_heads * kv_len * d_head   # QK dot + AV dot
    # Read Q (1 token) + K and V caches (kv_len tokens) + write output
    bytes_q      = 2 * batch * n_heads * d_head
    bytes_kv     = 2 * 2 * batch * n_heads * kv_len * d_head   # K and V
    bytes_out    = 2 * batch * n_heads * d_head
    total_bytes  = bytes_q + bytes_kv + bytes_out
    return flop / total_bytes

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]
kv_lengths  = [128, 512, 2048, 8192]

print("  Decode linear layer intensity  (FLOP/byte)  — varies with batch size")
print(f"  {'Batch':>8} | {'Intensity':>12} | {'Regime (A100)':>20}")
print("  " + "-"*46)
for b in batch_sizes:
    I = decode_linear_intensity(b)
    regime = "compute-bound ✅" if I >= A100.ridge_point else f"memory-bound  (need {A100.ridge_point/I:.0f}× more)"
    print(f"  {b:>8} | {I:>12.1f} | {regime}")

In [ ]:
# ── 📊 Heatmap: intensity as a function of (batch_size, kv_length) ────────────
batch_range  = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512])
kv_range     = np.array([64, 256, 512, 1024, 2048, 4096, 8192])

# Build intensity grid for the attention operation
grid = np.zeros((len(batch_range), len(kv_range)))
for i, b in enumerate(batch_range):
    for j, kv in enumerate(kv_range):
        grid[i, j] = decode_attention_intensity(b, kv)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#0f0f1a")

for ax, (data, title, cmap) in zip(axes, [
    (grid, "Attention Intensity (FLOP/byte)", "YlOrRd"),
    (np.log10(grid / A100.ridge_point + 1e-3),
     "log₁₀(Intensity / A100 Ridge Point)", "RdYlGn"),
]):
    ax.set_facecolor("#0f0f1a")
    im = ax.imshow(data, aspect="auto", cmap=cmap, origin="lower")
    cbar = plt.colorbar(im, ax=ax)
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")

    ax.set_xticks(range(len(kv_range)));   ax.set_xticklabels(kv_range, color="white", fontsize=8)
    ax.set_yticks(range(len(batch_range))); ax.set_yticklabels(batch_range, color="white", fontsize=8)
    ax.set_xlabel("KV Cache Length (tokens)", color="white", fontsize=11)
    ax.set_ylabel("Batch Size", color="white", fontsize=11)
    ax.set_title(title, color="white", fontsize=11, fontweight="bold")

    # Mark ridge boundary on the right panel
    if "log" in title:
        ax.contour(np.log10(grid / A100.ridge_point + 1e-3), levels=[0],
                   colors=["cyan"], linewidths=2)
        ax.text(4.5, 7.5, "Ridge\nboundary", color="cyan", fontsize=8,
                ha="center", va="top", fontweight="bold")

plt.suptitle("KV-Cached Decode: Arithmetic Intensity Landscape",
             color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("intensity_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
print("📊 Right panel: GREEN = compute-bound, RED = memory-bound.")
print("   Most realistic decode settings live in the red. That is the KV cache tax.")

## 5. 🔧 Your Turn

Now it is time for you to get your hands into the machinery. We are going to implement two functions from scratch.

### TODO 1: Compute the Effective Memory Bandwidth Utilisation

In the real world, we rarely achieve peak bandwidth. The metric we care about is **how much of the available bandwidth we are actually using**. Given a measured runtime and the bytes moved, compute the utilisation.

In [ ]:
def compute_bandwidth_utilisation(
    bytes_moved: float,
    runtime_seconds: float,
    peak_bandwidth_tbs: float,
) -> dict:
    """
    Compute how efficiently we are using available memory bandwidth.

    Parameters
    ----------
    bytes_moved        : Total bytes transferred to/from VRAM (float)
    runtime_seconds    : Wall-clock time for the operation in seconds (float)
    peak_bandwidth_tbs : Hardware peak bandwidth in TB/s (float)

    Returns
    -------
    A dict with keys:
        'achieved_bandwidth_tbs'  : float  — actual TB/s we achieved
        'utilisation_pct'         : float  — achieved / peak * 100
        'bottleneck'              : str    — "memory-bound" or "compute-bound"

    Implementation hints
    --------------------
    # Step 1: Compute achieved bandwidth in bytes/s, then convert to TB/s
    #         (1 TB = 1e12 bytes)
    # Step 2: Compute utilisation as a percentage of peak bandwidth
    # Step 3: For this function, always label the bottleneck "memory-bound"
    #         if utilisation > 50%, otherwise "compute-bound" (a simplification)
    """
    # ============ TODO ============
    # Step 1: achieved bandwidth in TB/s
    achieved_bandwidth_tbs = ???

    # Step 2: utilisation percentage
    utilisation_pct = ???

    # Step 3: bottleneck label
    bottleneck = ???
    # ============ END TODO ========

    return {
        "achieved_bandwidth_tbs": achieved_bandwidth_tbs,
        "utilisation_pct":        utilisation_pct,
        "bottleneck":             bottleneck,
    }

In [ ]:
# ✅ Verification cell — run this after implementing the function above
# If your implementation is correct, all assertions will pass silently.

# Test 1: moving 1 TB in 1 second on hardware with 2 TB/s peak → 50% utilisation
result = compute_bandwidth_utilisation(
    bytes_moved=1e12, runtime_seconds=1.0, peak_bandwidth_tbs=2.0)
assert abs(result["achieved_bandwidth_tbs"] - 1.0) < 1e-6, \
    f"Expected 1.0 TB/s, got {result['achieved_bandwidth_tbs']}"
assert abs(result["utilisation_pct"] - 50.0) < 1e-4, \
    f"Expected 50.0%, got {result['utilisation_pct']}"
assert result["bottleneck"] == "memory-bound", \
    f"Expected 'memory-bound', got {result['bottleneck']}"

# Test 2: 2 TB in 1 second → 100% utilisation, memory-bound
result2 = compute_bandwidth_utilisation(
    bytes_moved=2e12, runtime_seconds=1.0, peak_bandwidth_tbs=2.0)
assert abs(result2["utilisation_pct"] - 100.0) < 1e-4

# Test 3: 0.1 TB in 1 second → 5% utilisation, compute-bound
result3 = compute_bandwidth_utilisation(
    bytes_moved=0.1e12, runtime_seconds=1.0, peak_bandwidth_tbs=2.0)
assert result3["bottleneck"] == "compute-bound"

print("✅ All assertions passed! Your bandwidth utilisation function is correct.")
print(f"\n  Example: 1 TB moved in 1 s on A100:")
print(f"  Achieved bandwidth : {result['achieved_bandwidth_tbs']:.2f} TB/s")
print(f"  Utilisation        : {result['utilisation_pct']:.1f}%")
print(f"  Bottleneck         : {result['bottleneck']}")

### TODO 2: Model the KV Cache Memory Footprint

The KV cache grows with every token generated. Let us build a function that computes its total memory footprint given model and sequence parameters. This is essential for understanding the memory pressure the cache creates.

In [ ]:
def kv_cache_memory_bytes(
    n_layers:    int,
    n_heads:     int,
    d_head:      int,
    seq_len:     int,
    batch_size:  int,
    dtype_bytes: int = 2,       # FP16 = 2 bytes, FP32 = 4 bytes
) -> dict:
    """
    Compute the total memory footprint of the KV cache.

    For each layer, we store:
      - K matrix: [batch_size, n_heads, seq_len, d_head]
      - V matrix: [batch_size, n_heads, seq_len, d_head]

    Parameters
    ----------
    n_layers    : Number of transformer layers
    n_heads     : Number of attention heads per layer
    d_head      : Dimension of each attention head
    seq_len     : Current sequence length (tokens in KV cache)
    batch_size  : Number of sequences cached in parallel
    dtype_bytes : Bytes per element (2 for FP16, 4 for FP32)

    Returns
    -------
    dict with keys:
        'total_bytes'  : int   — total bytes consumed by the KV cache
        'total_gb'     : float — total gigabytes
        'per_token_bytes': float — bytes per token across all layers

    Hints
    -----
    # Step 1: Compute elements in one K or V tensor:
    #         elements = batch_size * n_heads * seq_len * d_head
    # Step 2: Total elements = 2 (K and V) * n_layers * elements
    # Step 3: Total bytes = total_elements * dtype_bytes
    # Step 4: per_token_bytes = total_bytes / seq_len  (bytes per token)
    """
    # ============ TODO ============
    # Step 1: elements in ONE K or V tensor
    elements_one = ???

    # Step 2: total elements (K + V, all layers)
    total_elements = ???

    # Step 3: total bytes
    total_bytes = ???

    # Step 4: bytes per token
    per_token_bytes = ???
    # ============ END TODO ========

    return {
        "total_bytes":      total_bytes,
        "total_gb":         total_bytes / 1e9,
        "per_token_bytes":  per_token_bytes,
    }

In [ ]:
# ✅ Verification cell
# GPT-2 Small: 12 layers, 12 heads, head_dim=64, FP16
result = kv_cache_memory_bytes(
    n_layers=12, n_heads=12, d_head=64,
    seq_len=1024, batch_size=1, dtype_bytes=2)

# Sanity: 12 layers × 2 (K+V) × 1 × 12 × 1024 × 64 × 2 bytes
expected = 12 * 2 * 1 * 12 * 1024 * 64 * 2
assert result["total_bytes"] == expected, \
    f"Expected {expected} bytes, got {result['total_bytes']}"
assert abs(result["total_gb"] - expected / 1e9) < 1e-6

# Per-token check
assert abs(result["per_token_bytes"] - expected / 1024) < 1e-3

print("✅ KV cache memory function verified!")
print()
print("  GPT-2 Small, seq_len=1024, batch=1, FP16:")
print(f"  Total KV cache : {result['total_gb']:.3f} GB")
print(f"  Per token      : {result['per_token_bytes']:.0f} bytes/token")

# Show how it scales for a larger model
print()
print("  LLaMA-2 7B scale (32 layers, 32 heads, head_dim=128):")
for slen in [512, 2048, 8192, 32768]:
    r = kv_cache_memory_bytes(32, 32, 128, slen, batch_size=1)
    print(f"    seq_len={slen:>6}: {r['total_gb']:.2f} GB")

## 6. Putting It All Together

Let us now combine everything — the roofline model, arithmetic intensity calculations, and KV cache footprint — into one unified analysis function and visualisation.

In [ ]:
def full_inference_analysis(
    model_name:   str,
    n_layers:     int,
    d_model:      int,
    n_heads:      int,
    seq_len:      int,       # context / KV cache length
    batch_size:   int = 1,
    dtype_bytes:  int = 2,
    hw:           HardwareSpec = A100,
) -> dict:
    """
    Compute the complete roofline picture for one decode step of a transformer.
    Returns a dict of all key metrics.
    """
    d_head = d_model // n_heads

    # ── KV cache footprint ──────────────────────────────────────────────────
    kv = kv_cache_memory_bytes(n_layers, n_heads, d_head,
                                seq_len, batch_size, dtype_bytes)

    # ── Arithmetic intensity for one decode step ────────────────────────────
    # We analyse the two dominant operations: linear layers and attention
    # For the full model there are 4 linear projections per layer (Q, K, V, O)
    # plus 2 in the MLP (we approximate MLP as 2 d→4d projections)
    total_flop_linear = 0
    total_bytes_linear = 0

    for _ in range(n_layers):
        for (d_in, d_out) in [(d_model, d_model)] * 4 + \
                              [(d_model, 4*d_model), (4*d_model, d_model)]:
            flop  = 2 * batch_size * d_in * d_out
            b_in  = dtype_bytes * batch_size * d_in
            b_w   = dtype_bytes * d_in * d_out
            b_out = dtype_bytes * batch_size * d_out
            total_flop_linear  += flop
            total_bytes_linear += b_in + b_w + b_out

    # Attention: one query token attending over seq_len keys/values
    attn_flop  = n_layers * 2 * batch_size * n_heads * seq_len * d_head
    attn_bytes = n_layers * dtype_bytes * (
        batch_size * n_heads * d_head          +  # Q
        2 * batch_size * n_heads * seq_len * d_head  +  # K, V (from cache)
        batch_size * n_heads * d_head          )  # output

    total_flop  = total_flop_linear + attn_flop
    total_bytes = total_bytes_linear + attn_bytes

    intensity = total_flop / total_bytes
    perf_bound = hw.achieved_performance(np.array([intensity]))[0]
    fraction_of_ridge = intensity / hw.ridge_point

    return dict(
        model_name         = model_name,
        kv_cache_gb        = kv["total_gb"],
        per_token_bytes    = kv["per_token_bytes"],
        total_flop_gflop   = total_flop / 1e9,
        total_bytes_gb     = total_bytes / 1e9,
        intensity          = intensity,
        perf_bound_tflops  = perf_bound,
        fraction_of_ridge  = fraction_of_ridge,
        regime             = "memory-bound" if intensity < hw.ridge_point else "compute-bound",
    )

# ── Run for several representative models ──────────────────────────────────────
models = [
    dict(model_name="GPT-2 Small (117M)",  n_layers=12, d_model=768,  n_heads=12),
    dict(model_name="GPT-2 Large (774M)",  n_layers=36, d_model=1280, n_heads=20),
    dict(model_name="LLaMA-2 7B (approx)", n_layers=32, d_model=4096, n_heads=32),
    dict(model_name="LLaMA-2 70B (approx)",n_layers=80, d_model=8192, n_heads=64),
]

print(f"{'Model':<28} {'KV (GB)':>9} {'Intensity':>12} {'% of Ridge':>12} {'Regime'}")
print("-" * 80)
for m in models:
    for slen in [512, 4096]:
        r = full_inference_analysis(**m, seq_len=slen, hw=A100)
        print(f"  {m['model_name']:<26} | KV_len={slen:<5} | "
              f"KV={r['kv_cache_gb']:>5.2f}GB | "
              f"I={r['intensity']:>6.2f} | "
              f"{r['fraction_of_ridge']*100:>5.1f}% | "
              f"{r['regime']}")
print()
print("  Every model, at every sequence length, is firmly memory-bound.")
print("  The KV cache makes inference faster, but keeps us deep in bandwidth territory.")

In [ ]:
# ── 📊 Side-by-side: intensity grows with batch, shrinks with seq_len ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor("#0f0f1a")

# Left: intensity vs batch size for LLaMA-7B, fixed seq_len=2048
batches = [1, 2, 4, 8, 16, 32, 64]
intensities_batch = [
    full_inference_analysis("LLaMA-7B", 32, 4096, 32,
                             seq_len=2048, batch_size=b)["intensity"]
    for b in batches
]

ax = axes[0]
ax.set_facecolor("#0f0f1a")
ax.semilogx(batches, intensities_batch, "o-", color="#ffa94d", lw=2.5, ms=8)
ax.axhline(A100.ridge_point, color="#00d4ff", lw=2, ls="--",
           label=f"A100 ridge ({A100.ridge_point:.0f})")
ax.fill_between(batches, intensities_batch, alpha=0.2, color="#ffa94d")
ax.set_xlabel("Batch Size", color="white", fontsize=12)
ax.set_ylabel("Arithmetic Intensity (FLOP/byte)", color="white", fontsize=12)
ax.set_title("Intensity vs Batch Size\n(LLaMA-7B, KV_len=2048)", color="white", fontsize=11)
ax.tick_params(colors="white")
for sp in ax.spines.values(): sp.set_edgecolor("#333355")
ax.legend(facecolor="#1a1a2e", labelcolor="white")
ax.set_xticks(batches); ax.set_xticklabels(batches, color="white")

# Right: intensity vs seq_len for LLaMA-7B, batch=1
seqlens = [128, 256, 512, 1024, 2048, 4096, 8192]
intensities_seq = [
    full_inference_analysis("LLaMA-7B", 32, 4096, 32,
                             seq_len=s, batch_size=1)["intensity"]
    for s in seqlens
]

ax = axes[1]
ax.set_facecolor("#0f0f1a")
ax.semilogx(seqlens, intensities_seq, "o-", color="#ff6b6b", lw=2.5, ms=8)
ax.axhline(A100.ridge_point, color="#00d4ff", lw=2, ls="--",
           label=f"A100 ridge ({A100.ridge_point:.0f})")
ax.fill_between(seqlens, intensities_seq, alpha=0.2, color="#ff6b6b")
ax.set_xlabel("KV Cache Length (tokens)", color="white", fontsize=12)
ax.set_ylabel("Arithmetic Intensity (FLOP/byte)", color="white", fontsize=12)
ax.set_title("Intensity vs Context Length\n(LLaMA-7B, batch=1)", color="white", fontsize=11)
ax.tick_params(colors="white")
for sp in ax.spines.values(): sp.set_edgecolor("#333355")
ax.legend(facecolor="#1a1a2e", labelcolor="white")
ax.set_xticks(seqlens); ax.set_xticklabels([str(s) for s in seqlens],
                                             color="white", rotation=30, fontsize=8)

plt.suptitle("The KV Cache Tax: Always Memory-Bound, Getting Worse with Context Length",
             color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("kv_cache_tax.png", dpi=120, bbox_inches="tight")
plt.show()
print("📊 As context length grows, arithmetic intensity actually DECREASES.")
print("   More KV cache to load → more bytes moved → worse intensity ratio.")
print("   This is the hidden cost materialising as a concrete number.")

## 7. Empirical Measurement — Timing Real Operations on GPU

Theory is satisfying. Numbers are better. Let us measure the actual memory bandwidth we achieve running a real linear layer on the GPU at different batch sizes, and see where we land on the roofline.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
print()

torch.manual_seed(42)

def measure_linear_intensity(batch: int, d: int = 768,
                              n_warmup: int = 20, n_trials: int = 100) -> dict:
    """
    Measure empirical arithmetic intensity for Y = X @ W on GPU.
    Returns theoretical intensity alongside measured bandwidth utilisation.
    """
    dtype = torch.float16
    X = torch.randn(batch, d, dtype=dtype, device=device)
    W = torch.randn(d, d,    dtype=dtype, device=device)

    # Theoretical quantities
    flop_theory   = 2 * batch * d * d
    bytes_theory  = 2 * (batch * d + d * d + batch * d)  # FP16
    intensity_theory = flop_theory / bytes_theory

    # Warm up
    for _ in range(n_warmup):
        _ = X @ W

    if device == "cuda":
        torch.cuda.synchronize()
        start_event = torch.cuda.Event(enable_timing=True)
        end_event   = torch.cuda.Event(enable_timing=True)
        start_event.record()
        for _ in range(n_trials):
            out = X @ W
        end_event.record()
        torch.cuda.synchronize()
        elapsed_ms = start_event.elapsed_time(end_event)
    else:
        import time
        t0 = time.perf_counter()
        for _ in range(n_trials):
            out = X @ W
        elapsed_ms = (time.perf_counter() - t0) * 1000

    elapsed_per_op_s = (elapsed_ms / 1000) / n_trials
    achieved_flops   = flop_theory / elapsed_per_op_s          # FLOP/s
    achieved_tflops  = achieved_flops / 1e12
    achieved_bw_tbs  = bytes_theory / elapsed_per_op_s / 1e12  # TB/s

    return dict(
        batch               = batch,
        intensity_theory    = intensity_theory,
        elapsed_us          = elapsed_per_op_s * 1e6,
        achieved_tflops     = achieved_tflops,
        achieved_bw_tbs     = achieved_bw_tbs,
    )

# ── Sweep batch sizes ──────────────────────────────────────────────────────────
print(f"  {'Batch':>6} | {'Intensity':>12} | {'Time (µs)':>10} | "
      f"{'Achieved (TFLOP/s)':>20} | {'Regime guess'}")
print("  " + "-"*75)

results = []
for b in [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]:
    r = measure_linear_intensity(b)
    results.append(r)
    regime = "compute-bound" if r["achieved_tflops"] > 10 else "memory-bound"
    print(f"  {b:>6} | {r['intensity_theory']:>12.2f} | {r['elapsed_us']:>10.1f} | "
          f"{r['achieved_tflops']:>20.2f} | {regime}")

In [ ]:
# ── 📊 Plot achieved performance on the actual roofline ───────────────────────
# Detect which GPU we are on and use its approximate spec
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    if "T4" in gpu_name:
        hw_measured = T4
    elif "A100" in gpu_name:
        hw_measured = A100
    else:
        # Generic fallback — scale T4 by a rough factor
        hw_measured = HardwareSpec(gpu_name, peak_compute_tflops=40, peak_bandwidth_tbs=0.4)
else:
    hw_measured = HardwareSpec("CPU (approximate)", peak_compute_tflops=1, peak_bandwidth_tbs=0.05)

intensity_arr = np.logspace(-2, 4, 1000)
roofline_perf = hw_measured.achieved_performance(intensity_arr)

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_facecolor("#0f0f1a"); fig.patch.set_facecolor("#0f0f1a")

ax.loglog(intensity_arr, roofline_perf, color="#00d4ff", lw=3,
          label=f"Theoretical roofline: {hw_measured.name}")
ax.axvline(hw_measured.ridge_point, color="#00d4ff", ls=":", lw=1.5, alpha=0.5)

# Plot measured points, coloured by batch size
sc = ax.scatter(
    [r["intensity_theory"] for r in results],
    [max(r["achieved_tflops"], 1e-3) for r in results],
    c=[r["batch"] for r in results],
    cmap="plasma", s=120, zorder=5, edgecolors="white", lw=0.5,
)
cbar = plt.colorbar(sc, ax=ax, label="Batch size")
cbar.set_label("Batch size", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")

# Annotate a few points
for r in results[::3]:
    ax.annotate(f"B={r['batch']}", (r["intensity_theory"], max(r["achieved_tflops"], 1e-3)),
                textcoords="offset points", xytext=(6, 4),
                color="white", fontsize=8)

ax.set_xlabel("Arithmetic Intensity (FLOP/byte)", color="white", fontsize=12)
ax.set_ylabel("Achieved Performance (TFLOP/s)",   color="white", fontsize=12)
ax.set_title(f"Empirical Roofline: Linear Layer on {hw_measured.name}",
             color="white", fontsize=13, fontweight="bold")
ax.tick_params(colors="white")
for sp in ax.spines.values(): sp.set_edgecolor("#333355")
ax.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=10)

plt.tight_layout()
plt.savefig("empirical_roofline.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"📊 Measured points on the {hw_measured.name} roofline.")
print("   Small batches cluster deep in the memory-bound slope.")
print("   This is exactly what happens during LLM single-token decode.")

## 8. 🎯 Final Output — The Complete Annotated Roofline

Let us now build the definitive, fully annotated roofline diagram that tells the complete story of this notebook in one image.

In [ ]:
# ── The complete, publication-quality annotated roofline ──────────────────────
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

np.random.seed(42)

fig, ax = plt.subplots(figsize=(13, 8))
ax.set_facecolor("#080814")
fig.patch.set_facecolor("#080814")

# ── Draw rooflines for both GPUs ──────────────────────────────────────────────
intensity = np.logspace(-2, 4, 2000)

for hw, col, ls in [(A100, "#00d4ff", "-"), (T4, "#a78bfa", "--")]:
    perf = hw.achieved_performance(intensity)
    ax.loglog(intensity, perf, color=col, lw=2.5, ls=ls,
              label=f"{hw.name}  (ridge={hw.ridge_point:.0f} FLOP/byte)", alpha=0.9)
    ax.axhline(hw.peak_compute_tflops, color=col, lw=0.8, ls=":", alpha=0.25)

# ── Shade regimes ─────────────────────────────────────────────────────────────
ax.axvspan(1e-2, A100.ridge_point, alpha=0.04, color="#ff6b6b")
ax.axvspan(A100.ridge_point, 1e4,  alpha=0.04, color="#69db7c")
ax.text(0.05, 0.8, "MEMORY-BOUND\n(bandwidth limited)", transform=ax.transAxes,
        color="#ff9999", fontsize=11, alpha=0.7, fontweight="bold")
ax.text(0.72, 0.8, "COMPUTE-BOUND\n(arithmetic limited)", transform=ax.transAxes,
        color="#99ff99", fontsize=11, alpha=0.7, fontweight="bold")

# ── Workload annotations ───────────────────────────────────────────────────────
workload_points = [
    # (intensity, label, colour, note)
    (0.035, "Naive inference\n(no KV cache, B=1)",
     "#ff4757", "Every token forces\nfull context re-read"),
    (0.3,   "KV-cached decode\n(B=1, short context)",
     "#ffa502", "Better — but still\nfirmly memory-bound"),
    (1.8,   "KV-cached decode\n(B=32, short context)",
     "#ff7f50", "Batch helps, but\nstill below ridge"),
    (45,    "Prefill / prompt\nprocessing (B=1, S=512)",
     "#2ed573", "Closer to ridge —\nnaturally batched"),
    (250,   "Training\n(large batch, full forward+back)",
     "#1e90ff", "Compute-bound ✅\nGPU fully utilised"),
]

for I, label, color, note in workload_points:
    y = min(A100.peak_bandwidth_tbs * I, A100.peak_compute_tflops)
    ax.scatter([I], [y], s=200, color=color, zorder=8,
               edgecolors="white", linewidths=1.0)
    ax.annotate(
        label,
        xy=(I, y), xycoords="data",
        xytext=(I * 2.2, y * 1.6),
        color=color, fontsize=8.5, fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.2,
                        connectionstyle="arc3,rad=-0.2"),
        bbox=dict(boxstyle="round,pad=0.3", fc="#0f0f1a", ec=color,
                  alpha=0.85, lw=1)
    )
    # Side note
    ax.text(I * 0.95, y * 0.55, note, color=color, fontsize=7,
            alpha=0.7, ha="right", va="top",
            path_effects=[pe.withStroke(linewidth=2, foreground="#080814")])

# ── KV cache memory annotations ───────────────────────────────────────────────
ax.text(0.98, 0.06,
        "KV Cache Memory Footprint (FP16, batch=1):\n"
        "  GPT-2 Small (12L), 2K ctx :  0.57 GB\n"
        "  LLaMA-2 7B  (32L), 2K ctx :  4.19 GB\n"
        "  LLaMA-2 7B  (32L), 8K ctx : 16.78 GB",
        transform=ax.transAxes, ha="right", va="bottom",
        color="#cccccc", fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.5", fc="#0f0f1a",
                  ec="#333355", alpha=0.9))

# ── Ridge point callout for A100 ──────────────────────────────────────────────
ax.axvline(A100.ridge_point, color="#00d4ff", lw=1.2, ls="--", alpha=0.4)
ax.text(A100.ridge_point * 1.08, 2,
        f"A100 ridge\n{A100.ridge_point:.0f} FLOP/byte",
        color="#00d4ff", fontsize=8, alpha=0.7, rotation=90, va="bottom")

# ── Labels and formatting ──────────────────────────────────────────────────────
ax.set_xlim(1e-2, 1e4)
ax.set_ylim(0.05, 2000)
ax.set_xlabel("Arithmetic Intensity  (FLOP / byte moved from VRAM)",
              color="white", fontsize=13)
ax.set_ylabel("Achieved Performance  (TFLOP/s)",
              color="white", fontsize=13)
ax.set_title(
    "The Roofline Model — Where LLM Inference Lives\n"
    "The KV Cache trades compute pressure for memory pressure",
    color="white", fontsize=14, fontweight="bold", pad=15)

ax.tick_params(colors="white", which="both")
ax.yaxis.set_tick_params(labelcolor="white")
ax.xaxis.set_tick_params(labelcolor="white")
for sp in ax.spines.values(): sp.set_edgecolor("#333355")

ax.grid(True, which="major", color="#1a1a2e", lw=0.8, alpha=0.7)
ax.grid(True, which="minor", color="#111122", lw=0.4, alpha=0.4)

ax.legend(facecolor="#0f0f1a", labelcolor="white", fontsize=10,
          loc="upper left", framealpha=0.9, edgecolor="#333355")

# ── Vizuara watermark ─────────────────────────────────────────────────────────
ax.text(0.01, 0.01, "Vizuara AI/ML Series — Section 4",
        transform=ax.transAxes, color="#555577", fontsize=8,
        fontstyle="italic")

plt.tight_layout()
plt.savefig("final_roofline_annotated.png", dpi=150, bbox_inches="tight",
            facecolor="#080814")
plt.show()
print("🎯 Final annotated roofline saved as 'final_roofline_annotated.png'")
print()
print("  This single diagram captures the key insight of Section 4:")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │  The KV cache eliminates recomputation, but the    │")
print("  │  memory it occupies — and the bandwidth needed to  │")
print("  │  load it every decode step — keeps LLM inference   │")
print("  │  firmly in the memory-bound region of the roofline.│")
print("  │  The shipyard sits idle, waiting for trucks.       │")
print("  └─────────────────────────────────────────────────────┘")

## 9. Reflection and Next Steps

Let us pause and make sure every piece of the picture is clear.

### 🧩 What We Built

We started with two abstract resources — arithmetic throughput and memory bandwidth — and turned them into a precise measurement framework. We derived arithmetic intensity from first principles, showed that a single-token decode step achieves roughly 1 FLOP/byte against a ridge point of 156 FLOP/byte on the A100, and measured this empirically on a real GPU. We showed how the KV cache, while eliminating the quadratic re-computation of naive inference, simultaneously creates a large, growing block of memory that must be read in full at every decode step — pushing us deeper into the memory-bound regime as context length increases.

### 🤔 Reflection Questions

**Question 1 — The batch size lever:**
Looking at the intensity-vs-batch-size plot, increasing batch size from 1 to 64 raises arithmetic intensity from ~1 to ~50 FLOP/byte. Why does batching help so much? What does it change about the ratio of arithmetic to bytes moved? Is there a limit to how much it can help?

**Question 2 — The context length trap:**
Longer context should be *better* for the model (more information). But our analysis shows that longer context makes the memory-bound problem *worse*. How would you explain this tension to a product manager who wants to maximise context window size? What engineering decisions would you need to make?

**Question 3 — The prefill/decode asymmetry:**
Prefill (processing the prompt) and decode (generating tokens) are two very different regimes. Prefill is naturally batched (many tokens at once) and sits close to the roofline. Decode is inherently sequential and sits deep in the memory-bound region. What does this suggest about how you might want to schedule compute differently for these two phases?

**Question 4 — The dtype lever:**
Our analysis assumed FP16 (2 bytes per element). Suppose we used FP8 (1 byte per element) for the KV cache — a technique used in production systems. Sketch what happens to the arithmetic intensity and the KV cache footprint. Does switching to FP8 move us out of the memory-bound region?

### 🏆 Optional Challenges

If you want to go deeper, here are three challenges that extend what we built:

**Challenge 1 — Multi-Query Attention (MQA) / Grouped-Query Attention (GQA):**
In standard multi-head attention, every head has its own K and V matrices. In MQA, all heads share a single K and V. Modify `kv_cache_memory_bytes` to accept an optional `n_kv_heads` parameter (where `n_kv_heads < n_heads` for GQA) and replot the memory footprint comparison. How much does LLaMA-2's use of GQA (with `n_kv_heads = n_heads // 4`) change the picture?

**Challenge 2 — Roofline for Speculative Decoding:**
In speculative decoding, a small draft model proposes $k$ tokens, and the large target model verifies all $k$ in one forward pass. This means the target model's decode step has an effective batch size of $k$. Add speculative decoding as an annotated point on the roofline diagram (assume $k=4$, draft model is GPT-2 Small, target is LLaMA-7B). Does speculation move us meaningfully closer to the ridge point?

**Challenge 3 — Continuous Batching Simulator:**
Production LLM serving engines use continuous batching — different requests may be at different positions in their KV caches, but their decode steps are batched together. Implement a simple simulator that tracks, for a pool of $N$ concurrent requests (each at a random position between 0 and `max_seq_len`), the effective batch size seen by the GPU and the resulting arithmetic intensity at each step. Plot how intensity evolves as requests complete and new ones arrive.

In [ ]:
# ── Summary: the key numbers from this notebook ───────────────────────────────
print("="*65)
print("  SECTION 4 SUMMARY — Key Numbers to Remember")
print("="*65)
print()
print("  Hardware (A100 BF16):")
print(f"    Peak compute    : {A100.peak_compute_tflops:>6.0f}  TFLOP/s")
print(f"    Peak bandwidth  : {A100.peak_bandwidth_tbs:>6.1f}  TB/s")
print(f"    Ridge point     : {A100.ridge_point:>6.0f}  FLOP/byte")
print()
print("  Arithmetic intensity of key workloads:")
print(f"    Single-token decode (B=1, linear layer)   : ~1     FLOP/byte")
print(f"    Single-token decode (B=1, attention)      : ~0.5   FLOP/byte")
print(f"    Decode with batch=32                      : ~30    FLOP/byte")
print(f"    Prefill (S=512)                           : ~50    FLOP/byte")
print(f"    Training (large batch)                    : ~250   FLOP/byte")
print()
print("  KV cache footprint (FP16, batch=1):")
for model, n_layers, d_model, n_heads in [
    ("GPT-2 Small", 12, 768, 12),
    ("LLaMA-2 7B ", 32, 4096, 32),
]:
    d_head = d_model // n_heads
    for slen in [1024, 4096]:
        r = kv_cache_memory_bytes(n_layers, n_heads, d_head, slen, 1)
        print(f"    {model}, KV_len={slen:>5}: {r['total_gb']:.2f} GB")
print()
print("  Core insight:")
print("    The KV cache trades O(S²) arithmetic for O(S) memory reads.")
print("    This is strictly better in FLOPs — but on real hardware,")
print("    the memory bandwidth cost keeps us memory-bound regardless.")
print("    The bottleneck moves, not disappears.")
print()
print("  Next section preview:")
print("    We will explore techniques that fight back against this")
print("    bottleneck: Flash Attention, paged KV caches, quantisation,")
print("    and speculative decoding — each a different strategy for")
print("    climbing toward the roofline ridge point.")
print("="*65)